In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
#excel file path
DATA_PATH = Path(r"C:\Users\kacie\OneDrive - University of Bristol\FoAI\Coursework\data cleaning\MedHealth_SYNTHEA_GroupCW_Dataset_Final.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Put the CSV file in the same folder as this notebook."
    )
        
raw = pd.read_csv(DATA_PATH)

print(f"Loaded {raw.shape[0]} rows and {raw.shape[1]} columns")
raw.head() #first 5 rows of data in the given excel file

raw.dtypes.value_counts() #count number of columns that have each data types

display(pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_n": raw.isna().sum(),
    "missing_percent": (100 * raw.isna().mean()).round(1),
    "unique_values": raw.nunique(dropna=True)
}).sort_values("missing_percent", ascending=False))

print("Exact duplicate rows:", raw.duplicated().sum())
display(raw.describe(include="all").T)

df = raw.copy()
date_cols = []
for c in df.columns:
    if c.endswith('_DATE') or c.startswith('PERIOD_'):
        date_cols.append(c)

print(date_cols)
display(raw[['BIRTH_DATE','PERIOD_START','PERIOD_END']].head())

for c in date_cols:
    df[c] = pd.to_datetime(df[c], format = '%d/%m/%Y') # tell panda that dates are in D/M/Y format, and change to Y/M/D format with .to_datetime
    
    
    
lost = {}
for c in date_cols:
    blanks_after = df[c].isna().sum()
    blanks_before = raw[c].isna().sum()
    blanks = blanks_after - blanks_before
    lost[c] = int(blanks)
print(lost) #to know any dates that cant be converted into Y/M/D format -> stayed (NaT) blanked

df[['BIRTH_DATE','PERIOD_START','PERIOD_END']].head()

Loaded 297 rows and 52 columns


,dtype,missing_n,missing_percent,unique_values
GLUCOSE_LAST_RECORDED,float64,176,59.3,107
GLUCOSE_DATE,str,176,59.3,101
CREATININE_LAST_RECORDED,float64,176,59.3,83
CREATININE_DATE,str,176,59.3,101
TRIGLYCERIDES_LAST_RECORDED,float64,128,43.1,96
TRIGLYCERIDES_DATE,str,128,43.1,97
HDL_CHOLESTEROL_DATE,str,128,43.1,97
HDL_CHOLESTEROL_LAST_RECORDED,float64,128,43.1,99
TOTAL_CHOLESTEROL_DATE,str,128,43.1,97
TOTAL_CHOLESTEROL_LAST_RECORDED,float64,128,43.1,99


Exact duplicate rows: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PATIENT_ID,297,100,00bcbfc6-c7b9-bd0e-3d95-071f9fba6321,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BIRTH_DATE,297,96,06/11/1979,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SEX,297,2,M,150,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RACE,297,3,white,242,NaN,NaN,NaN,NaN,NaN,NaN,NaN
YEAR,297.0,NaN,NaN,NaN,2024.0,0.817875,2023.0,2023.0,2024.0,2025.0,2025.0
PERIOD_START,297,6,01/01/2025,99,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PERIOD_END,297,4,31/12/2024,99,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RECORD_CUTOFF_DATE,297,10,23/08/2026,81,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENCOUNTER_SYMPTOM_RECORDS,297.0,NaN,NaN,NaN,0.296296,0.636518,0.0,0.0,0.0,0.0,5.0
ENCOUNTER_CHECK_UP_RECORDS,297.0,NaN,NaN,NaN,0.804714,1.795898,0.0,0.0,0.0,1.0,15.0


['BIRTH_DATE', 'PERIOD_START', 'PERIOD_END', 'RECORD_CUTOFF_DATE', 'HEIGHT_CM_DATE', 'WEIGHT_KG_DATE', 'PAIN_SCORE_DATE', 'TOTAL_CHOLESTEROL_DATE', 'HDL_CHOLESTEROL_DATE', 'TRIGLYCERIDES_DATE', 'GLUCOSE_DATE', 'CREATININE_DATE']


,BIRTH_DATE,PERIOD_START,PERIOD_END
0,20/01/1981,01/01/2023,31/12/2023
1,20/01/1981,01/01/2024,31/12/2024
2,20/01/1981,01/01/2025,31/12/2025
3,01/11/1972,01/01/2023,31/12/2023
4,01/11/1972,01/01/2024,31/12/2024


{'BIRTH_DATE': 0, 'PERIOD_START': 0, 'PERIOD_END': 0, 'RECORD_CUTOFF_DATE': 0, 'HEIGHT_CM_DATE': 0, 'WEIGHT_KG_DATE': 0, 'PAIN_SCORE_DATE': 0, 'TOTAL_CHOLESTEROL_DATE': 0, 'HDL_CHOLESTEROL_DATE': 0, 'TRIGLYCERIDES_DATE': 0, 'GLUCOSE_DATE': 0, 'CREATININE_DATE': 0}


,BIRTH_DATE,PERIOD_START,PERIOD_END
0,1981-01-20,2023-01-01,2023-12-31
1,1981-01-20,2024-01-01,2024-12-31
2,1981-01-20,2025-01-01,2025-12-31
3,1972-11-01,2023-01-01,2023-12-31
4,1972-11-01,2024-01-01,2024-12-31
